## Tydzień 2 Dzień 3

Teraz przechodzimy do większego poziomu szczegółowości:

1. Różne modele

2. Structured Outputs

3. Guardrails

**Ta wersja jest inna niż oryginał: zamiast frameworka OpenAI Agents SDK, budujemy dalej na mechanizmach z `2_lab2.pl.ipynb`, rozszerzając je o dwa nowe tematy tego dnia.** `run()`, `trace()` i `make_agent_tool()` są tu redefiniowane w uproszczonej wersji (bez obsługi handoffs - nieużywanej w tym labie).

Co nowego w tym notatniku:

- **Structured Outputs**: `output_type=EmailReview` z Agents SDK zastępuje `anthropic.messages.parse(output_format=EmailReview)` - sama klasa Pydantic (`EmailReview(BaseModel)`) zostaje bez zmian, bo Pydantic jest niezależny od dostawcy.
- **Guardrails**: `@output_guardrail` i automatyczne hooki Runnera nie mają odpowiednika w Anthropic - odtwarzamy je jako zwykłe, jawnie wywoływane funkcje Pythona. Notatnik sam to sugeruje ("prościej zaimplementować guardrails jawnie") - w naszej wersji to jedyny dostępny sposób.
- **Część 1 (różne modele)**: oryginał demonstruje 3 agentów na 3 różnych dostawcach (Gemini, Kimi przez OpenRouter, GPT-OSS przez Groq). Piotr świadomie spłaszczył to do jednego modelu, Claude Haiku 4.5, dla wszystkich trzech - to gubi demonstrację różnorodności modeli, ale upraszcza notatnik i nie wymaga dodatkowych kluczy API.
- **OPCJONALNY DODATEK: Sandbox Agents** - brak odpowiednika w Anthropic (lokalny harness wykonawczy głęboko zintegrowany z Agents SDK), sekcja zostaje jako materiał referencyjny w oryginalnym OpenAI, nieskonwertowana.
- **OPCJONALNY DODATEK: MCP** - Anthropic ma natywne wsparcie dla MCP, ale architektura jest inna (server-side, nie klient w Twoim procesie) - zobacz notatkę przy tej sekcji.

In [ ]:
# Importy tej wersji notatnika. Zamiast biblioteki agents (OpenAI Agents SDK) i openai importujemy wyłącznie natywny klient Anthropic, w wariancie asynchronicznym.
# pydantic zostaje bez zmian - BaseModel i Field są niezależne od dostawcy LLM, ta sama klasa posłuży w Części 2 (Structured Outputs) do anthropic.messages.parse().
# asyncio.iscoroutinefunction przyda się w handle_tool_calls() do rozróżnienia zwykłych (sync) narzędzi od agent-jako-narzędzie (async) - ten sam wzorzec co w 2_lab2.pl.ipynb.
# contextmanager i time odtwarzają trace() z 2_lab2.pl.ipynb (redefiniowane tutaj, bo każdy notatnik tego kursu jest samodzielnym projektem).
# json przyda się do serializacji wyników narzędzi zwracanych do Claude, tak jak we wcześniejszych notatnikach.

from dotenv import load_dotenv  # wczytuje zmienne środowiskowe (klucze API) z pliku .env
from anthropic import AsyncAnthropic  # asynchroniczny klient Anthropic - zastępuje framework agents i klienta openai.AsyncOpenAI
import os  # zmienne środowiskowe
import asyncio  # asyncio.iscoroutinefunction do rozróżnienia sync/async narzędzi w handle_tool_calls()
import json  # serializacja wyników narzędzi do formatu JSON
import time  # pomiar czasu wewnątrz ręcznego trace()
from contextlib import contextmanager  # dekorator do napisania trace() jako context managera
from pydantic import BaseModel, Field  # niezależne od dostawcy - te same klasy posłużą do Structured Outputs w Części 2

load_dotenv(override=True)  # ładuje .env i nadpisuje istniejące zmienne środowiskowe
anthropic = AsyncAnthropic()  # klucz brany z ANTHROPIC_API_KEY w .env; klient asynchroniczny, bo cały ten notatnik używa run() z await

MODEL = "claude-haiku-4-5"  # najtańszy dostępny model - pułap kosztowy na czas przechodzenia przez kurs

In [ ]:
# Sprawdzenie klucza API - w oryginale ta komórka sprawdzała cztery klucze (OpenAI, Google, OpenRouter, Groq), bo Część 1 pokazywała 3 różne modele za różnymi dostawcami.
# W tej wersji wszystkie trzy sales_agent są spłaszczone do jednego modelu Claude (patrz notatka na górze notatnika), więc potrzebny jest tylko jeden klucz.
# Wzorzec diagnostyczny (print z prefiksem klucza) jest identyczny jak w oryginale, tylko dla jednej zmiennej środowiskowej zamiast czterech.

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")  # klucz API Anthropic z pliku .env

if anthropic_api_key:  # sprawdź, czy zmienna w ogóle jest ustawiona
    print(f"Klucz API Anthropic istnieje i zaczyna się od {anthropic_api_key[:8]}")  # sanity check przeszedł, pokaż tylko prefiks
else:
    print("Klucz API Anthropic nie jest ustawiony")  # brak zmiennej w .env

In [ ]:
# Prompt systemowy agenta sprzedażowego - ten sam kontekst biznesowy co w 2_lab2.pl.ipynb (ComplAI, SOC2), ale bez podziału na 3 style pisania.
# Ta wersja notatnika skupia się na różnicy MODELI (Część 1), nie stylów - stąd jeden wspólny prompt zamiast trzech person.

instructions = """
Jesteś przedstawicielem handlowym pracującym dla ComplAI,
firmy dostarczającej narzędzie SaaS zapewniające zgodność z SOC2 i przygotowanie do audytów, oparte na AI.
Piszesz przekonujące e-maile sprzedażowe, które mają duże szanse na odpowiedź.
"""  # wspólny prompt systemowy dla wszystkich trzech (spłaszczonych) agentów

### Łatwo podłączyć dowolne modele przez endpointy kompatybilne z OpenAI, w 3 krokach:

KROK 1: Znajdź base URL kompatybilny z OpenAI (patrz Guide 9 w folderze guides)

KROK 2: Stwórz instancję klienta biblioteki Pythona (wersję async)

KROK 3: Stwórz obiekt modelu

**W tym notatniku pomijamy te trzy kroki** - Część 1 spłaszcza wszystkich trzech agentów do jednego modelu Claude (patrz notatka na górze), więc nie potrzebujemy osobnych klientów dla Gemini/OpenRouter/Groq. Ten przepis (podłączanie dowolnego dostawcy przez kompatybilny endpoint) wciąż jest prawdziwy i przydatny - pełny wzorzec dla natywnego Anthropic SDK (nie przez zgodność z OpenAI, tylko bezpośrednio) znajdziesz w `guides/09_ai_apis_and_ollama.ipynb`.

In [ ]:
# W oryginale sales_agent1/2/3 to trzy różne modele za różnymi dostawcami (Gemini, Kimi przez OpenRouter, GPT-OSS przez Groq) - to właśnie demonstrowała Część 1 tego labu.
# W tej wersji WSZYSTKIE trzy są spłaszczone do tego samego stringu instrukcji na Claude Haiku 4.5 - patrz notatka na górze notatnika, dlaczego i co to kosztuje (traci się demonstrację różnorodności modeli).
# Mechanizm poniżej (agent-jako-narzędzie, sales_manager wybierający i wysyłający) działa identycznie niezależnie od tego, czy agenci są na trzech różnych modelach, czy na jednym - resztę labu to nie zmienia.

sales_agent1 = instructions  # w oryginale: Gemini Sales Agent
sales_agent2 = instructions  # w oryginale: Kimi Sales Agent (przez OpenRouter)
sales_agent3 = instructions  # w oryginale: GPT-OSS Sales Agent (przez Groq)

### Mechanizmy z `2_lab2.pl.ipynb`, w uproszczonej wersji

Redefiniujemy `trace()`, `run()` i `make_agent_tool()` dokładnie tak, jak w `2_lab2.pl.ipynb` (każdy notatnik tego kursu jest samodzielnym projektem) - ale bez obsługi handoffs i `tool_choice`, bo ten lab ich nie potrzebuje. Sales Manager w tym notatniku (w przeciwieństwie do lab2) nie wymusza użycia narzędzia parametrem - polega wyłącznie na instrukcji w prompcie, dokładnie jak w oryginale.

In [ ]:
# Ta komórka definiuje trace() - dokładnie ten sam, lekki, lokalny odpowiednik obserwowalności co w 1_lab1.pl.ipynb i 2_lab2.pl.ipynb.
# @contextmanager pozwala napisać funkcję generatorową, którą Python zamienia w obiekt obsługujący "with trace(...): ...".
# Kod przed yield wykonuje się przy wejściu do bloku with, kod po yield - przy wyjściu z niego, nawet jeśli w środku wystąpi wyjątek.
# Redefiniujemy ją tutaj, a nie importujemy z innego notatnika, bo każdy notatnik tego kursu jest samodzielnym, niezależnym projektem.

@contextmanager
def trace(name: str):  # lokalny, uproszczony odpowiednik trace() z OpenAI Agents SDK - bez wysyłki danych na zewnątrz
    start = time.time()  # zapamiętaj moment startu, żeby policzyć czas trwania
    print(f"[trace] start: {name}")  # znacznik początku sekwencji wywołań
    yield  # tutaj wykonuje się kod wewnątrz bloku "with trace(...):"
    print(f"[trace] koniec: {name} ({time.time() - start:.2f}s)")  # znacznik końca razem z czasem trwania

In [ ]:
# Ta komórka to async odpowiednik Agent + Runner.run() z poprzednich notatników - uproszczony, bo ten lab nie potrzebuje handoffs ani tool_choice.
# run() jest "async def" i używa AsyncAnthropic (zainicjalizowanego w komórce z importami) - konsekwentnie z resztą notatnika, gdzie wszystkie wywołania są await.
# handle_tool_calls() wykonuje każde wywołanie narzędzia (sync albo async - agent-jako-narzędzie jest async, send_email_tool niżej jest sync) i buduje listę bloków tool_result.
# Ten jeden helper obsłuży całą Część 1 tego notatnika (Sales Manager wybierający i wysyłający e-mail przez narzędzia).

async def handle_tool_calls(tool_use_blocks: list) -> list[dict]:  # wykonuje wywołania narzędzi i buduje tool_result
    results = []  # lista bloków tool_result do wysłania z powrotem
    for block in tool_use_blocks:  # iteruj po każdym bloku tool_use z odpowiedzi
        tool = globals().get(block.name)  # znajdź funkcję Pythona o tej samej nazwie co narzędzie
        if tool is None:  # narzędzie o takiej nazwie nie istnieje w tym notatniku
            output = f"Nieznane narzędzie: {block.name}"  # komunikat błędu zwracany do Claude
        elif asyncio.iscoroutinefunction(tool):  # narzędzia typu agent-jako-tool są async (wywołują zagnieżdżone run())
            output = await tool(**block.input)  # wykonaj narzędzie asynchroniczne
        else:
            output = tool(**block.input)  # wykonaj zwykłe, synchroniczne narzędzie (np. send_email_tool)
        results.append({
            "type": "tool_result",  # Anthropic: blok tool_result zamiast wiadomości z rolą "tool" jak w OpenAI
            "tool_use_id": block.id,  # musi się zgadzać z id bloku tool_use, na który odpowiadamy
            "content": json.dumps(output),  # treść wyniku jako string JSON
        })
    return results  # zwracane bloki trafią razem do JEDNEJ wiadomości user


async def run(instructions: str, user_message: str, history: list | None = None, tools: list | None = None) -> tuple[str, list]:  # async odpowiednik Agent + Runner.run()
    messages = (history or []) + [{"role": "user", "content": user_message}]  # doklej nową wiadomość do historii (albo zacznij od zera)
    response = await anthropic.messages.create(
        model=MODEL, max_tokens=16000, system=instructions, messages=messages, tools=tools or [],
    )  # await, bo anthropic to teraz AsyncAnthropic
    while response.stop_reason == "tool_use":  # pętla trwa, dopóki Claude chce użyć narzędzia
        tool_use_blocks = [block for block in response.content if block.type == "tool_use"]  # wyciągnij bloki tool_use z odpowiedzi
        results = await handle_tool_calls(tool_use_blocks)  # wykonaj narzędzia i zbierz wyniki
        messages.append({"role": "assistant", "content": response.content})  # cała odpowiedź assistant wraca do historii
        messages.append({"role": "user", "content": results})  # wszystkie wyniki narzędzi w JEDNEJ wiadomości user
        response = await anthropic.messages.create(
            model=MODEL, max_tokens=16000, system=instructions, messages=messages, tools=tools or [],
        )  # kolejne zapytanie z zaktualizowaną historią
    text = next(block.text for block in response.content if block.type == "text")  # finalna odpowiedź tekstowa (content[0] bywa ThinkingBlock)
    messages.append({"role": "assistant", "content": response.content})  # zapisz finalną odpowiedź w historii do ewentualnego dalszego użycia
    return text, messages  # zwróć tekst (jak result.final_output) i historię (jak result.to_input_list())

In [ ]:
# Odpowiednik agent.as_tool(...) z OpenAI Agents SDK - zamienia innego agenta (jego instrukcje) w narzędzie wywoływane przez agenta nadrzędnego.
# Ta sama funkcja co w 2_lab2.pl.ipynb, tylko bez parametru handoff (nieużywanego w tym labie) - make_agent_tool() zwraca parę: schemat JSON i funkcję Pythona wywołującą zagnieżdżony run().
# Zwrócony tekst subagenta trafia z powrotem do agenta nadrzędnego jako zwykły tool_result - to jest sedno "agent jako narzędzie": kontrola wraca (A -> B -> A).
# Funkcja narzędzia jest asynchroniczna (async def), bo wywołuje await run(...) wewnątrz - stąd asyncio.iscoroutinefunction() w handle_tool_calls() wyżej.

def make_agent_tool(name: str, description: str, instructions: str, tools: list | None = None):  # buduje parę (schemat, funkcja) z instrukcji subagenta
    async def agent_tool(input: str) -> str:  # funkcja narzędzia - wywołuje subagenta i zwraca jego finalny tekst
        text, _ = await run(instructions, input, tools=tools)  # pełne, zagnieżdżone wywołanie run() na subagencie
        return text  # tekst subagenta trafia do Claude jako wynik narzędzia
    schema = {
        "name": name,  # nazwa narzędzia - musi się zgadzać z nazwą, pod którą zapiszesz zwróconą funkcję w globals()
        "description": description,  # opis czytany przez Claude przy decyzji, kiedy użyć tego narzędzia
        "input_schema": {
            "type": "object",
            "properties": {
                "input": {"type": "string", "description": "Instrukcja dla subagenta, np. treść zadania do wykonania"},
            },
            "required": ["input"],
            "additionalProperties": False,
        },
    }
    return schema, agent_tool  # para gotowa do wpisania do globals() (funkcja) i do listy tools= (schemat)

In [ ]:
# Zamiana trzech (spłaszczonych) sales_agent w narzędzia wywoływane przez agenta-menedżera - odpowiednik sales_agent1.as_tool(tool_name="sales_agent1", ...) x3.
# W oryginale nazwy narzędzi to "sales_agent1/2/3" (nie "sales_email_writer_N" jak w 2_lab2.pl.ipynb) - zachowujemy tę samą konwencję nazewnictwa co ten konkretny notatnik.
# Każda para (schemat, funkcja) z make_agent_tool() musi trafić do globals() pod nazwą zgodną z "name" w schemacie - stąd przypisanie do zmiennych sales_agent1_tool/2/3.

description = "Użyj tego narzędzia, żeby napisać e-mail sprzedażowy. W argumencie input po prostu poinstruuj je, żeby napisało e-mail sprzedażowy."  # opis czytany przez Claude - stąd po polsku, zgodnie z konwencją repo

tool1_json, sales_agent1_tool = make_agent_tool("sales_agent1", description, sales_agent1)  # narzędzie wywołujące pierwszego (spłaszczonego) agenta
tool2_json, sales_agent2_tool = make_agent_tool("sales_agent2", description, sales_agent2)  # narzędzie wywołujące drugiego (spłaszczonego) agenta
tool3_json, sales_agent3_tool = make_agent_tool("sales_agent3", description, sales_agent3)  # narzędzie wywołujące trzeciego (spłaszczonego) agenta

In [ ]:
# Import gotowych funkcji do wysyłki z messenger.py - ten moduł jest w 100% niezależny od dostawcy LLM (sam SMTP i Pushover), więc nie wymaga żadnej zmiany przy przejściu na Anthropic.
# USE_EMAIL to prosty przełącznik: True wymusza próbę wysyłki e-mailem, bez fallbacku na push (w przeciwieństwie do send_message() z 2_lab2.pl.ipynb, gdzie flaga była obliczana automatycznie z obecności kluczy).
# send_message() to ta sama funkcja-dyspozytor co w 2_lab2.pl.ipynb, tylko teraz reużywa send_email/push zaimportowane z osobnego modułu zamiast definiować je lokalnie w notatniku.

from messenger import send_email, push  # gotowe funkcje wysyłki - zdefiniowane raz w messenger.py, reużywane przez oba notatniki Tygodnia 2

USE_EMAIL = True  # wymuś próbę wysyłki e-mailem (bez automatycznego fallbacku na push)

def send_message(subject, text_body, html_body):  # wyślij wiadomość głównym kanałem albo fallbackiem
    if USE_EMAIL:  # jeśli USE_EMAIL jest ustawione na True
        send_email(subject, text_body, html_body)  # wyślij przez SMTP
    else:
        push(f"Subject: {subject}\n\n{text_body}")  # w przeciwnym razie wyślij push z tematem i treścią tekstową

In [ ]:
# Ręczne wywołanie testowe funkcji send_message() zdefiniowanej w komórce wyżej - sprawdza, że dane SMTP z .env faktycznie działają.
# Treść jest przetłumaczona na polski, bo to tekst, który faktycznie przeczytasz w skrzynce jako wiadomość testową.

send_message("Kolejny test", "Hura!", "<html><body><h1>Hura!</h1></body></html>")  # e-mail testowy

In [ ]:
# Teraz to samo narzędzie send_message, ale w formacie wywoływanym przez Claude - odpowiednik dekoratora @function_tool z OpenAI Agents SDK.
# @function_tool w oryginale automatycznie generuje schemat JSON z sygnatury funkcji i docstringa (Args: ...) - w Anthropic nie ma takiej magii, schemat piszemy ręcznie, dokładnie jak w 2_lab2.pl.ipynb.
# send_email_tool_json opisuje narzędzie: nazwę, opis czytany przez Claude i schemat argumentów (subject, text_body, html_body) w formacie JSON Schema.
# Sama funkcja send_email_tool wywołuje już istniejące send_message() i zwraca string ze statusem - Claude dostanie ten string jako tool_result.
# Nazwa funkcji musi się zgadzać z kluczem "name" w schemacie, bo nasza pętla run() szuka narzędzia po nazwie przez globals().get(block.name).

send_email_tool_json = {
    "name": "send_email_tool",  # nazwa narzędzia - musi się zgadzać z nazwą funkcji Pythona
    "description": "Wyślij e-mail o podanym temacie i treści do wszystkich potencjalnych klientów sprzedażowych",  # opis czytany przez Claude
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI, i nie generuje go automatycznie z docstringa
        "type": "object",
        "properties": {
            "subject": {"type": "string", "description": "Temat e-maila"},
            "text_body": {"type": "string", "description": "Treść e-maila jako czysty tekst"},
            "html_body": {"type": "string", "description": "Treść e-maila w formacie HTML"},
        },
        "required": ["subject", "text_body", "html_body"],
        "additionalProperties": False,
    },
}


def send_email_tool(subject: str, text_body: str, html_body: str) -> str:  # wersja send_message() jako narzędzie Claude
    send_message(subject, text_body, html_body)  # wyślij e-mailem (USE_EMAIL=True, bez fallbacku)
    return "E-mail wysłany pomyślnie"  # ten string trafi do Claude jako tool_result

In [ ]:
# Zbieramy wszystkie narzędzia w jedną listę: trzy narzędzia wywołujące (spłaszczonych) sales_agent i jedno narzędzie do wysyłki.
# Ta lista trafi jako tools= do run() w kolejnej komórce - Sales Manager sam zdecyduje, kiedy i które z nich wywołać.

tools = [tool1_json, tool2_json, tool3_json, send_email_tool_json]  # komplet narzędzi dostępnych agentowi-menedżerowi niżej

In [ ]:
# Agent-menedżer, który samodzielnie decyduje, kiedy wywołać które narzędzie - dokładnie jak Sales Manager z 2_lab2.pl.ipynb, ale tym razem BEZ wymuszenia tool_choice.
# W oryginale ten agent też nie ma ModelSettings(tool_choice="required") - polega wyłącznie na instrukcji w treści task, że ma użyć narzędzia do wysyłki.
# instructions to krótki opis roli, task to szczegółowa instrukcja krok po kroku, przekazywana jako user_message do run().

instructions = """
Jesteś Menedżerem Sprzedaży w ComplAI. Twoim celem jest znalezienie najlepszego zimnego e-maila sprzedażowego, korzystając z narzędzi sales_agent.
"""  # rola agenta-menedżera

task = """
Wykonaj następujące kroki:

1. Wygeneruj wersje robocze: użyj każdego z trzech narzędzi sales_agent, żeby wygenerować różne wersje e-maila.
Po prostu poinstruuj każde z nich, żeby napisało e-mail sprzedażowy; nie są potrzebne dalsze szczegóły.
Nie kontynuuj, dopóki wszystkie trzy wersje nie będą gotowe, po jednej z każdego narzędzia.

2. Oceń i wybierz: przejrzyj wersje robocze i wybierz jeden, najlepszy e-mail, kierując się własnym osądem, który będzie najskuteczniejszy.

3. Użyj swojego narzędzia, żeby wysłać najlepszy e-mail (i tylko najlepszy) do użytkownika. Wyślij tylko 1 e-mail.
"""  # szczegółowa instrukcja krok po kroku dla agenta-menedżera

sales_manager = instructions  # "Sales Manager" - string instrukcji

In [ ]:
# Uruchomienie agenta-menedżera w bloku trace() - Claude sam decyduje, kiedy wywołać które z czterech narzędzi.
# print(result) jest CELOWO poza blokiem with, dokładnie jak w oryginale (print(result.final_output) też jest poza with trace(...)).

with trace("Menedżer sprzedaży na różnych modelach"):  # jeden znacznik czasu obejmujący całą pracę menedżera
    result, _ = await run(sales_manager, task, tools=tools)  # menedżer sam decyduje o kolejności wywołań narzędzi
print(result)  # wypisz finalną odpowiedź menedżera

## Nie ma tu prawdziwego "trace" do obejrzenia

Tak jak w poprzednich notatnikach: nasz lokalny `trace()` wypisuje tylko dwie linie w konsoli (start/koniec + czas trwania) nad tą komórką - nie zapisuje niczego na `platform.openai.com/traces` ani żadnej innej platformie.

## Część 2: Structured Outputs

LLM domyślnie generuje tekst w języku naturalnym. Ale możemy sprawić, żeby zamiast tego wygenerował "obiekt Pythona".

Osiąga się to zwykłą sztuczką: sprytnymi promptami i JSON-em!

1. Określamy obiekt Pythona
2. W prompcie systemowym LLM dostaje instrukcję, żeby odpowiedzieć w JSON i trzymać się schematu reprezentującego ten obiekt Pythona
3. LLM zwraca JSON, a framework/biblioteka na tej podstawie tworzy obiekt Pythona

Kiedy określamy obiekt Pythona, tworzymy podklasę BaseModel, która jest częścią frameworka Pydantic.

Pydantic to framework, który łatwo pozwala zdefiniować schemat JSON i mapowanie między Pythonem a JSON-em.

UWAGI:

1. W sposobie, w jaki się to robi, jest coś naprawdę sprytnego - jeśli Cię to interesuje, poszukaj "constrained decoding".
2. Nie wszyscy dostawcy wspierają Structured Outputs.

In [ ]:
# Definicja obiektu Pythona (schematu), który ma wypełnić Claude - klasa Pydantic, w 100% niezależna od dostawcy LLM, zostaje bez zmian względem oryginału.
# Field(description=...) to tekst czytany przez model przy wypełnianiu schematu, więc tłumaczymy go na polski, tak samo jak prompty i opisy narzędzi.
# Ta sama klasa posłuży w Części 3 (Guardrails) do sprawdzenia wygenerowanych e-maili - stąd jej nazwa i pola są ogólne, nie specyficzne dla jednego użycia.

class EmailReview(BaseModel):  # schemat odpowiedzi, który Claude ma wypełnić
    is_professional: bool = Field(description="Czy e-mail jest profesjonalny i odpowiedni")  # pole boolowskie
    number_of_sentences: int = Field(description="Liczba zdań w treści e-maila, nie licząc powitania i podpisu")  # pole liczbowe
    contains_placeholders: bool = Field(description="Czy e-mail zawiera placeholdery do personalizacji")  # pole boolowskie

In [ ]:
# Podgląd schematu JSON wygenerowanego automatycznie przez Pydantic z klasy EmailReview - identyczne dla OpenAI i Anthropic, bo to czysty Pydantic, bez udziału żadnego SDK dostawcy.

EmailReview.model_json_schema()  # schemat JSON odpowiadający klasie EmailReview

In [ ]:
# Przykładowy, celowo słaby e-mail sprzedażowy - służy jako dane wejściowe do sprawdzenia przez agenta-checkera w kolejnych komórkach.
# Treść przetłumaczona na polski (to tekst, który przeczytałby prawdziwy odbiorca), celowo zawiera placeholder [imię] i niezbyt profesjonalny ton.

email = """
Cześć [imię],

Piszę do Ciebie, żeby zapytać, czy chciałbyś kupić nasz produkt. Jest naprawdę świetny. Będziesz żałować, jeśli go nie kupisz.

Na razie.

Ed
"""  # celowo niedopracowany e-mail: nieformalny ton + placeholder [imię]

In [ ]:
# Jedna, stabilna funkcja check_email() - używana trzykrotnie w tym notatniku (tu w Części 2, potem w guardrailu i w ręcznym sprawdzeniu w Części 3), zamiast osobnej konwersji per komórka.
# W oryginale to ten sam obiekt checker = Agent(..., output_type=EmailReview), wywoływany trzykrotnie przez Runner.run(checker, ...) - tutaj to jedna funkcja Pythona zamiast jednego obiektu Agent.
# anthropic.messages.parse(output_format=EmailReview) automatycznie generuje schemat JSON z klasy Pydantic i waliduje odpowiedź względem niego - odpowiednik output_type= z Agents SDK.
# response.parsed_output to gotowy, zwalidowany obiekt EmailReview (odpowiednik result.final_output przy output_type=EmailReview).
# checker_instructions to ten sam prompt co instructions="You review potential sales emails" z oryginalnego Agent(name="Checker", ...).

checker_instructions = "Sprawdzasz potencjalne e-maile sprzedażowe"  # prompt systemowy agenta-checkera

async def check_email(email: str) -> EmailReview:  # jedna, reużywalna funkcja - patrz komentarz wyżej o trzech miejscach użycia
    response = await anthropic.messages.parse(
        model=MODEL, max_tokens=16000, system=checker_instructions,
        messages=[{"role": "user", "content": email}],
        output_format=EmailReview,
    )  # messages.parse() automatycznie generuje schemat JSON z EmailReview i waliduje odpowiedź względem niego
    return response.parsed_output  # gotowy, zwalidowany obiekt EmailReview

In [ ]:
# Pierwsze użycie check_email() - sprawdź przykładowy e-mail z komórki wyżej i wyświetl wynik.
# review to teraz zwykły obiekt EmailReview (nie result.final_output jak w oryginale) - check_email() już zwraca gotowy, sparsowany obiekt.

review = await check_email(email)  # odpowiednik: result = await Runner.run(checker, email); review = result.final_output
review  # podgląd sparsowanego obiektu EmailReview

In [ ]:
# Podgląd pojedynczego pola obiektu EmailReview - zwykły dostęp do atrybutu Pydantic, identyczny niezależnie od dostawcy LLM.

review.is_professional  # dostęp do pojedynczego pola sparsowanego obiektu

## Część 3: Guardrails

Guardrails są niezwykle ważne w Agentic AI. Mówiąc prosto, to kontrole, które kodujesz albo w logice, albo przez kolejne wywołanie LLM, żeby zapobiec niepożądanemu zachowaniu.

Dla mnie implementacja Guardrails w OpenAI Agents SDK trochę przypomina "magię frameworka". Podejrzewam, że ich motywacją było pokazanie kontroli na poziomie frameworka, żeby zaadresować ten ważny temat.

Ale prościej i czyściej jest zaimplementować guardrails jawnie, jako osobne wywołania Runner.run(), albo sprawdzenia wewnątrz implementacji narzędzi.

Niezależnie od tego - zobaczmy narzędzia frameworka.

https://openai.github.io/openai-agents-python/guardrails/

**W naszej wersji notatnika nie ma frameworka, więc od razu implementujemy guardrails jawnie - dokładnie tak, jak sugeruje ten akapit.** Poniżej zobaczysz najpierw odtworzenie mechanizmu z Agents SDK (`check_guardrail()`), a potem sekcję "Z drugiej strony", która w oryginale pokazywała "prostszą" alternatywę - w naszej wersji obie sekcje wychodzą niemal identyczne, bo nie mamy nic bardziej "frameworkowego" do porównania.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Uwaga na pułapkę</h2>
            <span style="color:#ff7800;">W OpenAI Agents SDK są 3 typy Guardrails: wejściowe, wyjściowe i narzędziowe. Guardrails wejściowe uruchamiają się tylko dla pierwszego wejścia do pierwszego Agenta w Runner.run(). Guardrails wyjściowe uruchamiają się tylko dla finalnego wyjścia ostatniego agenta. Jeśli masz guardrails na innych agentach, nigdy nie zostaną wywołane.<br/><br/>
            Ta pułapka dotyczy wyłącznie automatycznego, ukrytego uruchamiania guardrails przez framework na podstawie POZYCJI agenta w łańcuchu Runner.run(). W naszej wersji, gdzie każde wywołanie check_guardrail() jest jawne w kodzie Pythona, ten problem nie istnieje - guardrail uruchamia się dokładnie tam, gdzie go wywołasz, niezależnie od tego, który to agent z kolei.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Odtworzenie mechanizmu output_guardrail z OpenAI Agents SDK - jako zwykła funkcja Pythona wywoływana JAWNIE przez nasz kod, nie automatyczny hook frameworka.
# check_guardrail() reużywa check_email() z Części 2 (ta sama funkcja, trzecie użycie w tym notatniku) - sprawdza tę samą parę kryteriów: profesjonalizm i obecność placeholderów.
# Zwraca bool: True oznacza "tripwire triggered" (odpowiednik tripwire_triggered=True z GuardrailFunctionOutput) - wiadomość NIE powinna zostać wysłana.
# W oryginale tripwire_triggered=True rzuca wyjątek OutputGuardrailTripwireTriggered, przerywając Runner.run() na twardo; tutaj świadomie wybieramy miękkie zachowanie (zwrot bool, decyzja należy do wywołującego kodu) - masz pełną kontrolę, bo to zwykła funkcja, nie automatyczny hook.
# W przeciwieństwie do oryginalnego @output_guardrail, ta funkcja nie ma dostępu do żadnego automatycznego kontekstu Runnera (ctx, agent) - dostaje po prostu tekst do sprawdzenia.

async def check_guardrail(message: str) -> bool:  # True = wiadomość nie powinna zostać wysłana
    review = await check_email(message)  # reużyj check_email() zdefiniowane w Części 2
    return review.contains_placeholders or not review.is_professional  # ten sam warunek co is_problem w oryginalnym email_guardrail

In [ ]:
# Persona "kowboj" - te same instrukcje sprzedażowe z doklejonym stylem mówienia. Ta sama technika co w 2_lab2.pl.ipynb (intro + styl).

cowboy_instructions = instructions + "\nMów jak kowboj"  # doklej styl do istniejących instrukcji

sales_agent_cowboy = cowboy_instructions  # "Cowboy" - w naszej wersji to sam string instrukcji; output_guardrails=[...] z oryginału nie istnieje, sprawdzenie robimy jawnie niżej

In [ ]:
# Wygeneruj e-mail w stylu kowboja, a potem JAWNIE sprawdź guardrail - w OpenAI Agents SDK ten krok wykonałby się automatycznie po zakończeniu agenta.
# email, _ = await run(...) - druga wartość krotki (historia) tutaj nie jest potrzebna, stąd _.
# check_guardrail() zwraca True, jeśli e-mail nie przeszedł sprawdzenia - wtedy nie wypisujemy/nie wysyłamy go dalej.

email, _ = await run(sales_agent_cowboy, "Napisz zimny e-mail sprzedażowy")  # wygeneruj e-mail w stylu kowboja
if await check_guardrail(email):  # jawne sprawdzenie guardrail (patrz komórka wyżej)
    print("Guardrail zadziałał: e-mail nie jest profesjonalny albo zawiera placeholdery - nie zostanie wysłany")
else:
    print(email)  # e-mail przeszedł sprawdzenie - wypisz go

### Sprawdź trace

Tak jak wyżej - nie ma tu prawdziwej platformy trace do obejrzenia, tylko lokalny znacznik czasu w konsoli.

## Z drugiej strony..

Mówiąc oczywistą rzecz: to jest prostsze i zadziała w dowolnym frameworku.

In [ ]:
# Ta sama persona "kowboj", tym razem bez żadnego wywołania check_guardrail() na tym etapie - guardrail sprawdzimy w kolejnej komórce, osobno.

simple_cowboy = cowboy_instructions  # "Simple Cowboy" - ten sam string instrukcji co sales_agent_cowboy wyżej
email, _ = await run(simple_cowboy, "Napisz zimny e-mail sprzedażowy")  # zwykłe wywołanie run(), bez żadnego mechanizmu guardrail
print(email)  # wypisz wygenerowany e-mail

In [ ]:
# Ręczne sprawdzenie tego samego e-maila, tym razem bez owijania w check_guardrail() - dokładnie ten sam warunek, tylko wypisany wprost w if/else zamiast schowany w funkcji.
# To trzecie i ostatnie użycie check_email() w tym notatniku.

review = await check_email(email)  # reużyj check_email() z Części 2
if not review.is_professional or review.contains_placeholders:  # ten sam warunek co w check_guardrail() wyżej, tym razem jawnie
    print("E-mail nie jest profesjonalny albo zawiera placeholdery i nie zostanie wysłany")
else:
    print("E-mail jest dobry")

## Sprawdź trace

Tak jak wyżej - nie ma tu prawdziwej platformy trace do obejrzenia, tylko lokalny znacznik czasu w konsoli.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ćwiczenie</h2>
            <span style="color:#ff7800;">• Wypróbuj różne modele<br/>• Dodaj więcej guardraili wejściowych i wyjściowych<br/>• Użyj structured outputs do generowania e-maila
            </span>
        </td>
    </tr>
</table>

## OPCJONALNY DODATEK: Sandbox Agents

Ten przykład zadziała tylko na Windows + WSL2, albo Mac, albo Linux.

https://openai.github.io/openai-agents-python/sandbox_agents/

To harness wykonawczy - runtime - "trwała przestrzeń robocza, w której agent może przeszukiwać duże zbiory dokumentów, edytować pliki, uruchamiać komendy, generować artefakty i wracać do pracy z zapisanego stanu sandboksa."

Trzeba skonfigurować:
1. Manifest: przestrzeń roboczą
2. Capabilities: co agent może robić
3. SandboxRunConfig: gdzie to działa

**Ta sekcja NIE MA odpowiednika w Anthropic SDK i nie została skonwertowana.** Sandbox Agents to głęboko zintegrowany z OpenAI Agents SDK harness wykonawczy (lokalny system plików, izolacja procesu, zapisywany stan) - zbudowanie własnego odpowiednika byłoby osobnym, dużym projektem infrastrukturalnym, nie konwersją API. Komórki poniżej zostają w oryginalnej postaci OpenAI jako materiał referencyjny - nie uruchomią się bez `openai-agents` i `OPENAI_API_KEY`, których celowo nie trzymasz w tym repo.

In [ ]:
# BEZ KONWERSJI - patrz notatka wyżej: Sandbox Agents nie ma odpowiednika w Anthropic. Poniższe komórki (39-46 oryginału) zostają w oryginalnej, angielskiej postaci OpenAI, jako materiał referencyjny.
# Nie uruchomią się bez zainstalowanego pakietu openai-agents i klucza OPENAI_API_KEY, których celowo nie trzymasz w tym repo.

from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [ ]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [ ]:
CODE_DIR

In [ ]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [ ]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

In [ ]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [ ]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini", default_manifest=manifest, capabilities=capabilities)

In [ ]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

## OPCJONALNY DODATEK: Zajawka MCP!

**Uwaga architektoniczna: to NIE jest ten sam mechanizm 1:1, mimo że obie strony wspierają MCP.** `MCPServerStreamableHttp` z OpenAI Agents SDK to klient MCP działający W TWOIM PROCESIE - biblioteka `agents` sama łączy się z serwerem MCP (stąd `async with ... as server` w oryginale). Konektor MCP w Anthropic jest SERVER-SIDE: to infrastruktura Anthropic łączy się z serwerem MCP, nie Twój kod - dlatego w wersji niżej nie ma żadnego klienta ani `async with` do otwarcia, tylko dwa dodatkowe parametry (`mcp_servers=`, `tools=[{"type": "mcp_toolset", ...}]`) w wywołaniu `messages.create()`. Efekt końcowy jest podobny, ale architektura jest inna - to nie jest kwestia innego SDK robiącego to samo.

In [ ]:
# Pytanie o bardzo świeżą (hipotetyczną, z przyszłości względem daty treningu modelu) funkcję SandboxAgents - Claude prawdopodobnie NIE będzie tego znał bez dostępu do aktualnej dokumentacji.
# Treść przetłumaczona na polski, bo to prompt wysyłany do modelu.

task = """
W nowej funkcji SandboxAgents w OpenAI Agents SDK, dostępnej od maja 2026, jaka jest rola obiektu Manifest?
Zawsze bądź dokładny/a. Jeśli nie znasz odpowiedzi, powiedz to wprost.
"""

In [ ]:
# "Przed": zwykłe wywołanie run() bez żadnego dostępu do zewnętrznej wiedzy - odpowiednik Agent(name="Expert", instructions="Answer the question", model="gpt-4o-mini").
# Oczekiwany wynik: Claude powinien uczciwie przyznać, że nie zna szczegółów tej (fikcyjnej, przyszłej) funkcji, zamiast zmyślać.

expert_instructions = "Odpowiedz na pytanie"  # ten sam prompt co instructions="Answer the question" z oryginału
answer, _ = await run(expert_instructions, task)  # zwykłe wywołanie run(), bez żadnego narzędzia ani dostępu do MCP
print(answer)  # oczekuj przyznania się do niewiedzy, nie zmyślonej odpowiedzi

In [ ]:
# "Po": to samo pytanie, tym razem z dostępem do serwera MCP Context7 (aktualna dokumentacja bibliotek) - patrz notatka architektoniczna wyżej.
# mcp_servers= podaje adres i nazwę serwera MCP; tools= z wpisem typu mcp_toolset odwołuje się do tej nazwy - oba parametry są wymagane razem, pominięcie któregoś zwraca błąd walidacji.
# Wymaga bety mcp-client-2025-11-20, więc wywołujemy anthropic.beta.messages.create(...), nie zwykłe anthropic.messages.create(...) - to jedyna komórka w tym notatniku poza run()/trace(), która woła klienta bezpośrednio.
# Claude sam decyduje, czy i kiedy skorzystać z narzędzia MCP - wynik trafia do response.content jako dodatkowe bloki, ale finalny tekst nadal wyciągamy tym samym wzorcem next(...) co wszędzie indziej w tym repo.

response = await anthropic.beta.messages.create(
    model=MODEL, max_tokens=16000,
    betas=["mcp-client-2025-11-20"],  # beta wymagana przez konektor MCP
    system="Użyj Context7, żeby odpowiedzieć na pytanie",  # odpowiednik instructions="Use Context7 to answer the question"
    mcp_servers=[{"type": "url", "url": "https://mcp.context7.com/mcp", "name": "context7"}],  # adres i nazwa serwera MCP - połączenie nawiązuje serwer Anthropic, nie ten notebook
    tools=[{"type": "mcp_toolset", "mcp_server_name": "context7"}],  # odwołanie do serwera po nazwie - wymagane razem z mcp_servers
    messages=[{"role": "user", "content": task}],  # to samo pytanie co w komórce "przed"
)
answer = next(block.text for block in response.content if block.type == "text")  # finalna odpowiedź tekstowa (content bywa dłuższe o bloki narzędzia MCP)
print(answer)  # tym razem oczekuj konkretnej odpowiedzi opartej na aktualnej dokumentacji

## Zobacz też traces:

Tak jak wyżej - nie ma tu prawdziwej platformy trace do obejrzenia, tylko lokalny znacznik czasu w konsoli (jeśli opakujesz powyższe wywołanie w `trace(...)`).